In [ ]:
# import the library
from pynq import Overlay     # import the overlay
from pynq import allocate    # import for CMA (contingeous memory allocation)
from pynq import DefaultIP   # import the ip connector library for extension
from pynq import Interrupt
import asyncio
import numpy as np
import os
import dfx4ml.cap         as cap
import dfx4ml.mem_alloc   as dataAlloc  # import the memory allocation library
import dfx4ml.dfx_unified as dfx_unified
import time

In [ ]:
PRJ_DIR = os.getcwd()
PRJ_HW_DIR = PRJ_DIR + '/hw/'
PRJ_TC_DIR = PRJ_DIR + '/sw/'
DFX_CONFIG_FILE = 'dfx_ctrl_meta.txt'

FULL_BS_NAME = 'system.bin'
PAR_BS_NAME_0 = 'par0.bin'  ###### dma to magic stream 1
INPUT_DATA_NAME = "input_x.npy"

AMT_QUERY = 100
INPUT_SHAPE = (AMT_QUERY, 1)  # 4*4* float32 = 64 bytes
OUTPUT_SHAPE = (AMT_QUERY, 1)  # 4*4* float32 = 64 bytes
AMT_SLOT = 1

input_x = np.arange(AMT_QUERY, dtype=np.int32).reshape(INPUT_SHAPE)
np.save(os.path.join(PRJ_TC_DIR, INPUT_DATA_NAME), input_x)


In [2]:
cap.change_pl_config_mode("pcap", True, "")

CHANGE CMD STDOUT: 0xFFCA3008 0xFFFFFFFF 0x1

CHANGE CMD ERROR : 
--------------------------------
TRIGGER CMD STDOUT: 0xFFCA3008

TRIGGER CMD ERROR : 
--------------------------------
READ CMD STDOUT: 0x1

READ CMD ERROR : 
--------------------------------


In [3]:
#### load the overlay
overlay  = Overlay(PRJ_HW_DIR + FULL_BS_NAME)

In [4]:
#### create the interrupt pin
overlay.interrupt_pins


{'magicSeq/MagicSeqTopIntr_0/hw_intr': {'controller': 'axi_intc_0',
  'index': 0,
  'fullpath': 'magicSeq/MagicSeqTopIntr_0/hw_intr'},
 'axi_intc_0/intr': {'controller': 'axi_intc_0',
  'index': 0,
  'fullpath': 'axi_intc_0/intr'}}

In [5]:
my_interrupt = Interrupt('dfx_unified_0/dfx_intr')  # index 0 from your mapping

In [ ]:
dfx_unifed_ip = overlay.dfx_unified_0

In [6]:
#### get the device
dmaIp      = dfx_unifed_ip.dfx_dma
dfx_ctrl  = dfx_unifed_ip.dfx_ctrl
dfx_mng = dfx_unifed_ip.dfx_mng

In [7]:
#### configure the dfx controller ip to match the address space
dfx_ctrl.config(PRJ_HW_DIR + DFX_CONFIG_FILE)
print("regIdxSize = ", dfx_ctrl.BLS_REGID)

regbank detect index (7, 8)
regbank detect index (2, 6)
regIdxSize =  5


In [12]:
inputX = np.load(PRJ_TC_DIR + INPUT_DATA_NAME)
if(inputX.shape != INPUT_SHAPE):
    raise Exception(f"inputX shape is {inputX} expect {INPUT_SHAPE}")
print("-------------init all data buffer -------------")
buf_input   , buf_input_phya   , buf_input_sz    = dataAlloc.alloc_data_uint(alloc_shape= INPUT_SHAPE , alloc_type=np.int32, input_x = inputX)
buf_out     , buf_out_phy      , buf_out_sz      = dataAlloc.alloc_data_uint(alloc_shape= OUTPUT_SHAPE, alloc_type=np.int32)
buf_input.flush()

print("------------- after init magic seq------")
print(dfx_mng.print_debug())

------ before init magic seq------
----- MAIN STATUS ------------------
--------> STATUS =  SHUTDOWN
--------> MAINCNT =  0
--------> ENDCNT  =  0
--------> DMAADDR  =  0x0
--------> DFXADDR  =  0x0
--------> INTR_ENA =  0x0
--------> INTR     =  0x0
--------> ROUND_TRIP =  0x0
----- SLOT DATA ------------------
------> slot 0 :
        srcAddr   : 0x0,  srcSize   : 0x0
        desAddr   : 0x0,  desSize   : 0x0
        status    : 0x0
        profileCnt: 0x0
        loadMask  : 0b0
        storeMask : 0b0
        stIntrMask: 0b0
------> slot 1 :
        srcAddr   : 0x0,  srcSize   : 0x0
        desAddr   : 0x0,  desSize   : 0x0
        status    : 0x0
        profileCnt: 0x0
        loadMask  : 0b0
        storeMask : 0b0
        stIntrMask: 0b0
------> slot 2 :
        srcAddr   : 0x0,  srcSize   : 0x0
        desAddr   : 0x0,  desSize   : 0x0
        status    : 0x0
        profileCnt: 0x0
        loadMask  : 0b0
        storeMask : 0b0
        stIntrMask: 0b0
------> slot 3 :
      

In [ ]:
dmaIp.reset()
dmaIp.init()
dmaIp.send(buf_input_phya, buf_input_sz)
dmaIp.recv(buf_out_phy   , buf_out_sz)

In [24]:
buf_out.invalidate()
np_parRes = np.array(buf_out, dtype=np.int32)
print(np_parRes)

[[[[0.41015625]
   [0.4296875 ]
   [0.3955078 ]
   [0.5       ]]

  [[0.453125  ]
   [0.3955078 ]
   [0.453125  ]
   [0.5       ]]

  [[0.5       ]
   [0.48828125]
   [0.39941406]
   [0.5       ]]

  [[0.4765625 ]
   [0.4609375 ]
   [0.5       ]
   [0.484375  ]]]


 [[[0.47265625]
   [0.47265625]
   [0.4296875 ]
   [0.5       ]]

  [[0.44921875]
   [0.48828125]
   [0.46484375]
   [0.5       ]]

  [[0.48828125]
   [0.39941406]
   [0.44140625]
   [0.5       ]]

  [[0.5       ]
   [0.48046875]
   [0.42578125]
   [0.48046875]]]


 [[[0.40722656]
   [0.4296875 ]
   [0.4375    ]
   [0.4921875 ]]

  [[0.4609375 ]
   [0.4140625 ]
   [0.4609375 ]
   [0.5       ]]

  [[0.4765625 ]
   [0.36621094]
   [0.43359375]
   [0.5       ]]

  [[0.5       ]
   [0.48828125]
   [0.43359375]
   [0.46484375]]]


 ...


 [[[0.3876953 ]
   [0.44921875]
   [0.4033203 ]
   [0.4609375 ]]

  [[0.36621094]
   [0.4296875 ]
   [0.5       ]
   [0.5       ]]

  [[0.48046875]
   [0.39160156]
   [0.3203125 ]
   [0.5       ]

In [25]:
print(f"Compare input_x and np_parRes: {np.array_equal(input_x, np_parRes)}")

[[[[2.07733765e-01]
   [9.68818069e-01]
   [1.71564773e-01]
   [2.17555314e-01]]

  [[8.66356552e-01]
   [2.19541356e-01]
   [7.65057564e-01]
   [9.92457211e-01]]

  [[3.66488963e-01]
   [6.78366303e-01]
   [7.42028534e-01]
   [2.25073352e-01]]

  [[7.87249729e-02]
   [1.10976942e-01]
   [7.41845965e-01]
   [3.99327248e-01]]]


 [[[5.56711793e-01]
   [5.37615657e-01]
   [4.36398864e-01]
   [8.57473671e-01]]

  [[2.30146684e-02]
   [2.56007642e-01]
   [8.88902009e-01]
   [2.28493825e-01]]

  [[2.80501932e-01]
   [2.61796802e-01]
   [3.83014768e-01]
   [4.70255792e-01]]

  [[1.64166600e-01]
   [6.98053896e-01]
   [5.22391349e-02]
   [3.30212176e-01]]]


 [[[5.63937485e-01]
   [1.12613011e-02]
   [9.84813273e-01]
   [5.17411649e-01]]

  [[6.62973300e-02]
   [6.92672551e-01]
   [3.90291631e-01]
   [9.32351172e-01]]

  [[4.23798747e-02]
   [6.19248196e-04]
   [3.94393414e-01]
   [3.13410789e-01]]

  [[1.47238225e-01]
   [6.25923038e-01]
   [2.94986814e-01]
   [1.36400625e-01]]]


 ...


 [[

In [26]:
print(f"source address {hex(dmaIp.get_mm2s_source_addr())}")
print(f"des address {hex(dmaIp.get_s2mm_dest_addr())}")

store element  32736  load element  32736


In [27]:
stream1 = overlay.dataMovement.streamDbg_1
print("store element ", stream1.read(0), " load element ",stream1.read(8))

store element  32736  load element  32736


In [28]:
stream2 = overlay.dataMovement.streamDbg_2
print("store element ", stream2.read(0), " load element ", stream2.read(8))

stream 1 status:  0  stream 2 status:  0


In [29]:
streamStatus = overlay.dataMovement.streamDbg_state
print("stream 1 status: ", streamStatus.read(0)& 0xF, " stream 2 status: ", (streamStatus.read(0)>> 4) & 0xF)


dma write status:  4098


In [30]:
print("dma write status: ", dmaIp.read(0x34))

In [ ]:
np.save("zcuOutput1023.npy", np_parRes)

In [ ]:
overlay.ip_dict["dfx_unified_0"]["phys_addr"]